# OutBoxML: модель 2 (TARGET_FREQ / TARGET_SEV)

Конфиги генерируются из артефактов train_loop_new (фичи + HPO). Калибровка isotonic — **снаружи** DSM.

**Прод-ансамбль** = refit на train ∪ 85% более старого test, isotonic на **15% самых свежих** дат test.

A/B raw vs cal: ECE + порог + финэффект на **одной** шкале proba (порог не копируется со старой шкалы).

Parity-fit нужен только для таблиц финэффекта на полном holdout Test (как collect C3). Старый example.ipynb / config_*_3 не трогаем.

На сервисе сырьё: перед prepare_dataset применить замороженный DQ из integration/results/querulus_dq_bounds_{version}.json (apply_frozen_dq_bounds). JSON фич — только encoding (UPPER, ПРОЧИЕ, _MEDIAN_).


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
PROJECT_ROOT = next(
    p for p in (_here, *_here.parents) if (p / "pyproject.toml").exists()
)
SRC = PROJECT_ROOT / "src"
OUTBOXML_ROOT = PROJECT_ROOT.parent.parent
for _p in (SRC, OUTBOXML_ROOT, PROJECT_ROOT):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
print("PROJECT_ROOT", PROJECT_ROOT)
print("OUTBOXML_ROOT", OUTBOXML_ROOT)


In [ ]:
import json
import pickle
import warnings
from copy import deepcopy

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")
pd.options.display.float_format = "{:,.2f}".format

from outboxml.core.prepared_datasets import PrepareDataset
from outboxml.data_subsets import DataPreprocessor
from outboxml.datasets_manager import DataSetsManager

from querulus.training.build_outboxml_configs import (
    dataframe_for_dsm,
    default_model_version,
    prepare_datasets_from_config,
    ensure_predictable_model,
    unwrap_estimator,
    write_outboxml_configs,
)
from querulus.training.outboxml_metrics import display_dsm_collect_metrics
from querulus.training.calibration import (
    compare_calibrator_ab,
    expected_calibration_error,
    fit_probability_calibrator,
)
from querulus.features.data_quality import write_service_dq_bounds
from querulus.fin_effect import (
    create_summary_table,
    export_business_html,
    print_best_threshold_report,
    resolve_fin_effect_config,
    run_fin_effect_pipeline,
)


In [ ]:
MODEL_VERSION = default_model_version(business="2", increment="v1")
DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "df_final_3.parquet"
RESULTS_DIR = PROJECT_ROOT / "integration" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("MODEL_VERSION", MODEL_VERSION)
print("DATASET_PATH", DATASET_PATH)

In [ ]:
df = dataframe_for_dsm(pd.read_parquet(DATASET_PATH))
print("df", df.shape)
built = write_outboxml_configs(
    df,
    version=MODEL_VERSION,
    parquet_path=str(DATASET_PATH.as_posix()),
)
periods = built["periods"]
CF_NAME = built["cf_name"]
RG_NAME = built["rg_name"]
display(Markdown("### Периоды (даты / n / n_positive по TARGET_FREQ)"))
display(periods["table"])
print("cutoff prod_cal", periods["prod_cutoff"])
print("cf", built["cf_path"])
print("rg", built["rg_path"])
print("cf_prod", built["cf_prod_path"])
print("rg_prod", built["rg_prod_path"])

In [ ]:
def _patch_dsm_models(dsm):
    for name, res in dsm.get_result().items():
        res.model = ensure_predictable_model(res.model)


def _prepared_X(dsm, model_name, data):
    result = dsm.get_result()[model_name]
    preproc = DataPreprocessor(
        prepare_dataset_interface_dict={
            model_name: PrepareDataset(model_config=result.model_config, check_prepared=False)
        },
        dataset=data.copy(),
        data_config=dsm.data_config,
        prepare_engine="pandas",
    )
    subset = preproc.get_subset(model_name, from_pickle=False)
    num = list(result.data_subset.features_numerical or [])
    cat = list(result.data_subset.features_categorical or [])
    cols = [c for c in num + cat if c in subset.X.columns]
    return subset.X[cols]


def _predict_cf(dsm, model_name, data, calibrator=None):
    X = _prepared_X(dsm, model_name, data)
    est = unwrap_estimator(dsm.get_result()[model_name].model)
    if calibrator is not None:
        proba = np.asarray(calibrator.predict_proba(X)[:, 1], dtype=float)
    elif hasattr(est, "predict_proba"):
        proba = np.asarray(est.predict_proba(X)[:, 1], dtype=float)
    else:
        proba = np.asarray(est.predict(X), dtype=float)
    return pd.Series(proba, index=X.index, dtype=float)


def _predict_rg(dsm, model_name, data):
    X = _prepared_X(dsm, model_name, data)
    est = unwrap_estimator(dsm.get_result()[model_name].model)
    pred = np.asarray(est.predict(X), dtype=float)
    return pd.Series(pred, index=X.index, dtype=float)


def _fit_isotonic_on_index(dsm, model_name, df_all, index, y_col="TARGET_FREQ"):
    data = df_all.loc[index]
    X = _prepared_X(dsm, model_name, data)
    y = pd.to_numeric(df_all.loc[X.index, y_col], errors="coerce").fillna(0).astype(int)
    n_pos = int((y == 1).sum())
    if n_pos < 200:
        print(f"[warn] isotonic: n_positive={n_pos} < 200 (n={len(y)})")
    est = unwrap_estimator(dsm.get_result()[model_name].model)
    proba_raw = np.asarray(est.predict_proba(X)[:, 1], dtype=float)
    ece_before = expected_calibration_error(y, proba_raw)
    calibrator = fit_probability_calibrator(est, X, y, method="isotonic")
    proba_cal = np.asarray(calibrator.predict_proba(X)[:, 1], dtype=float)
    ece_after = expected_calibration_error(y, proba_cal)
    print(f"isotonic {model_name}: n={len(y)} n_pos={n_pos} ECE {ece_before:.4f} → {ece_after:.4f}")
    return calibrator


def _fin_effect_table(df_all, index, proba, sev, *, threshold=None, title=""):
    cfg = resolve_fin_effect_config(
        df_all,
        frequency_target="TARGET_FREQ",
        severity_target="TARGET_SEV",
    )
    aligned = df_all.loc[index]
    fe = run_fin_effect_pipeline(
        aligned,
        proba.reindex(index),
        sev.reindex(index),
        aligned["TARGET_FREQ"],
        threshold=threshold,
        config=cfg,
    )
    if title:
        display(Markdown(f"### {title}"))
    print_best_threshold_report(fe)
    summary = create_summary_table(fe.frame, cfg)
    display(summary.style.format("{:,.0f}", subset=summary.columns[3:]))
    return fe, summary

## Parity: DSM на train_core∪val, test = полный holdout

Эти модели **не** идут в прод-pickle. Нужны для таблицы финэффекта на том же Test, что collect C3.


In [ ]:
from configs import config as querulus_outboxml_config

dsm_cf = DataSetsManager(
    config_name=str(built["cf_path"]),
    external_config=querulus_outboxml_config,
    prepared_datasets=prepare_datasets_from_config(built["cf_path"]),
)
dsm_cf.load_dataset(data=df)
dsm_cf.fit_models()
_patch_dsm_models(dsm_cf)

dsm_rg = DataSetsManager(
    config_name=str(built["rg_path"]),
    external_config=querulus_outboxml_config,
    prepared_datasets=prepare_datasets_from_config(built["rg_path"]),
)
dsm_rg.load_dataset(data=df)
dsm_rg.fit_models()
_patch_dsm_models(dsm_rg)
display_dsm_collect_metrics(dsm_cf, CF_NAME, task_type="classification", title=f"parity {CF_NAME}")
display_dsm_collect_metrics(dsm_rg, RG_NAME, task_type="regression", title=f"parity {RG_NAME}")
print("parity fit done", CF_NAME, RG_NAME)


In [ ]:
cal_idx = periods["splits"].cal
test_idx = periods["splits"].test
calibrator_parity = _fit_isotonic_on_index(dsm_cf, CF_NAME, df, cal_idx)
proba_test_raw = _predict_cf(dsm_cf, CF_NAME, df.loc[test_idx], calibrator=None)
proba_test_cal = _predict_cf(dsm_cf, CF_NAME, df.loc[test_idx], calibrator=calibrator_parity)
proba_test = proba_test_cal  # основная ветка для сверки/HTML: с калибратором
sev_test = _predict_rg(dsm_rg, RG_NAME, df.loc[test_idx])

_fe_cfg_ab = resolve_fin_effect_config(
    df, frequency_target="TARGET_FREQ", severity_target="TARGET_SEV"
)
ab_parity = compare_calibrator_ab(
    df,
    test_idx,
    proba_test_raw,
    proba_test_cal,
    sev_test,
    df.loc[test_idx, "TARGET_FREQ"],
    config=_fe_cfg_ab,
    title="parity A/B on full Test (cal fit on Cal, eval on Test)",
)
display(ab_parity.table)

## Таблицы финэффекта (сверка глазами)

1. Collect после HPO на полном Test — если в ядре есть `fin_effect_b`.
2. OutBoxML parity A/B: raw vs cal на полном Test (порог на каждой шкале свой).
3. Детальный отчёт ниже — на **cal** proba (как в экспорте с калибратором).


In [ ]:
_fe_collect = globals().get("fin_effect_b")
if _fe_collect is not None:
    display(Markdown("### Collect HPO / блок C3 (полный Test)"))
    print_best_threshold_report(_fe_collect)
    _cfg_c = globals().get("FIN_EFFECT_CONFIG_B")
    if _cfg_c is not None:
        _sum_c = create_summary_table(_fe_collect.frame, _cfg_c)
        display(_sum_c.style.format("{:,.0f}", subset=_sum_c.columns[3:]))
else:
    print("fin_effect_b нет в kernel — прогон collect C3 или смотри только таблицу OutBoxML ниже")

fe_parity_raw, _ = _fin_effect_table(
    df,
    test_idx,
    proba_test_raw,
    sev_test,
    title="OutBoxML parity RAW (полный Test, порог на raw proba)",
)
fe_parity, _ = _fin_effect_table(
    df,
    test_idx,
    proba_test_cal,
    sev_test,
    title="OutBoxML parity CAL (полный Test, порог на cal proba)",
)
# Согласованность: fe из A/B == детальные таблицы
assert abs(fe_parity.best_threshold - ab_parity.fe_cal.best_threshold) < 1e-9
assert abs(fe_parity_raw.best_threshold - ab_parity.fe_raw.best_threshold) < 1e-9

_fe_html = _fe_collect if _fe_collect is not None else fe_parity
_cfg_html = globals().get("FIN_EFFECT_CONFIG_B")
if _cfg_html is None:
    _cfg_html = resolve_fin_effect_config(
        df, frequency_target="TARGET_FREQ", severity_target="TARGET_SEV"
    )
_html_path = export_business_html(
    _fe_html,
    _cfg_html,
    path=PROJECT_ROOT / "notebooks" / "fin_effect_detailed.html",
    subtitle=(
        "Collect, блок финансового эффекта на полном Test"
        if _fe_collect is not None
        else "OutBoxML parity CAL proba (порог на калиброванных вероятностях), полный Test"
    ),
)
print(f"HTML для бизнеса: {_html_path}")

## Prod-refit (идёт в ансамбль)

Обучение: исходный train + test **до cutoff** (85% более старых строк test по дате).
Isotonic на **15% самых свежих** дат test.
A/B raw vs cal на том же freshest хвосте (fit+eval калибратора на одном срезе — в title явно).
Финэффект/порог для экспорта — на **cal** proba.


In [ ]:
dsm_cf_prod = DataSetsManager(
    config_name=str(built["cf_prod_path"]),
    external_config=querulus_outboxml_config,
    prepared_datasets=prepare_datasets_from_config(built["cf_prod_path"]),
)
dsm_cf_prod.load_dataset(data=df)
dsm_cf_prod.fit_models()
_patch_dsm_models(dsm_cf_prod)

dsm_rg_prod = DataSetsManager(
    config_name=str(built["rg_prod_path"]),
    external_config=querulus_outboxml_config,
    prepared_datasets=prepare_datasets_from_config(built["rg_prod_path"]),
)
dsm_rg_prod.load_dataset(data=df)
dsm_rg_prod.fit_models()
_patch_dsm_models(dsm_rg_prod)

prod_cal_idx = df.index[
    (pd.to_datetime(df[periods["date_column"]], errors="coerce") >= pd.Timestamp(periods["prod_test_period"][0]))
    & (pd.to_datetime(df[periods["date_column"]], errors="coerce") <= pd.Timestamp(periods["prod_test_period"][1]))
]
print("prod_cal n", len(prod_cal_idx))
calibrator_prod = _fit_isotonic_on_index(dsm_cf_prod, CF_NAME, df, prod_cal_idx)
proba_prod_raw = _predict_cf(dsm_cf_prod, CF_NAME, df.loc[prod_cal_idx], calibrator=None)
proba_prod_cal = _predict_cf(dsm_cf_prod, CF_NAME, df.loc[prod_cal_idx], calibrator=calibrator_prod)
sev_prod_cal = _predict_rg(dsm_rg_prod, RG_NAME, df.loc[prod_cal_idx])

ab_prod = compare_calibrator_ab(
    df,
    prod_cal_idx,
    proba_prod_raw,
    proba_prod_cal,
    sev_prod_cal,
    df.loc[prod_cal_idx, "TARGET_FREQ"],
    config=globals().get("_fe_cfg_ab")
    or resolve_fin_effect_config(
        df, frequency_target="TARGET_FREQ", severity_target="TARGET_SEV"
    ),
    title="prod A/B on 15% freshest test (cal fit+eval on same slice)",
)
display(ab_prod.table)

fe_prod_raw, _ = _fin_effect_table(
    df,
    prod_cal_idx,
    proba_prod_raw,
    sev_prod_cal,
    title="Prod-refit RAW (15% freshest, порог на raw)",
)
fe_prod, _ = _fin_effect_table(
    df,
    prod_cal_idx,
    proba_prod_cal,
    sev_prod_cal,
    title="Prod-refit CAL (15% freshest, порог на cal — для экспорта)",
)
assert abs(fe_prod.best_threshold - ab_prod.fe_cal.best_threshold) < 1e-9

display_dsm_collect_metrics(
    dsm_cf_prod, CF_NAME, task_type="classification", title=f"prod {CF_NAME}"
)
display_dsm_collect_metrics(
    dsm_rg_prod, RG_NAME, task_type="regression", title=f"prod {RG_NAME}"
)


## Export pickle (prod-модели)


In [ ]:
cf_export = dsm_cf_prod.get_result()[CF_NAME].dict_for_prod_export()
rg_export = dsm_rg_prod.get_result()[RG_NAME].dict_for_prod_export()
cf_export["model"] = ensure_predictable_model(cf_export["model"])
rg_export["model"] = ensure_predictable_model(rg_export["model"])

cf_pkl = RESULTS_DIR / f"querulus_cf_for_prod_{MODEL_VERSION}.pickle"
rg_pkl = RESULTS_DIR / f"querulus_rg_for_prod_{MODEL_VERSION}.pickle"
ans_pkl = RESULTS_DIR / f"querulus_ansamble_{MODEL_VERSION}.pickle"
cal_pkl = RESULTS_DIR / f"querulus_cf_calibrator_{MODEL_VERSION}.pickle"
dq_report = PROJECT_ROOT / "data" / "processed" / "data_quality_report.json"
dq_bounds_pkl = RESULTS_DIR / f"querulus_dq_bounds_{MODEL_VERSION}.json"
if dq_report.exists():
    write_service_dq_bounds(
        dq_bounds_pkl,
        model_version=MODEL_VERSION,
        report_path=dq_report,
    )
    print("dq_bounds", dq_bounds_pkl)
else:
    print("[warn] нет data_quality_report.json — dq_bounds не записан")
    dq_bounds_pkl = None

cf_pkl.write_bytes(pickle.dumps([cf_export]))
rg_pkl.write_bytes(pickle.dumps([rg_export]))
ensemble = [deepcopy(cf_export), deepcopy(rg_export)]
ans_pkl.write_bytes(pickle.dumps(ensemble))
cal_pkl.write_bytes(pickle.dumps(calibrator_prod))

meta = {
    "model_version": MODEL_VERSION,
    "periods": {
        k: list(v) if isinstance(v, tuple) else v
        for k, v in periods.items()
        if k in {
            "parity_train_period",
            "parity_test_period",
            "prod_train_period",
            "prod_test_period",
            "prod_cutoff",
            "date_column",
            "cal_period",
        }
    },
    "cf_name": CF_NAME,
    "rg_name": RG_NAME,
    "best_threshold": float(fe_prod.best_threshold),
    "best_threshold_branch": "cal",
    "calibrator_ab": {
        "parity_test": ab_parity.table.reset_index().to_dict(orient="records"),
        "prod_cal": ab_prod.table.reset_index().to_dict(orient="records"),
    },
    "dq_bounds": str(dq_bounds_pkl) if dq_bounds_pkl else None,
    "artifacts": {
        "cf": str(cf_pkl),
        "rg": str(rg_pkl),
        "ensemble": str(ans_pkl),
        "calibrator": str(cal_pkl),
    },
}
(RESULTS_DIR / f"querulus_meta_{MODEL_VERSION}.json").write_text(
    json.dumps(meta, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)
print("wrote", cf_pkl.name, rg_pkl.name, ans_pkl.name, cal_pkl.name,
      dq_bounds_pkl.name if dq_bounds_pkl else None)
print("best_threshold (cal)=", meta["best_threshold"])
print("calibrator_ab prod:\n", ab_prod.table)

## DQ bounds для сервиса

Файл querulus_dq_bounds_{version}.json — заборы с **сборки** df_final_3 (тот же winsorize, на котором учили).

**Контракт FastAPI (позже):** сырой вектор → apply_frozen_dq_bounds → prepare_dataset / predict. IQR на заявке не считать.


## Roundtrip predict


In [ ]:
sample = df.loc[prod_cal_idx].head(20)
res_cf = dsm_cf_prod.model_predict(sample, CF_NAME)
res_rg = dsm_rg_prod.model_predict(sample, RG_NAME)
p_cf = _predict_cf(dsm_cf_prod, CF_NAME, sample, calibrator=calibrator_prod)
p_rg = _predict_rg(dsm_rg_prod, RG_NAME, sample)
display(
    pd.DataFrame(
        {
            "dsm_cf": np.asarray(res_cf.prediction).ravel()[: len(sample)],
            "proba_cal": p_cf,
            "dsm_rg": np.asarray(res_rg.prediction).ravel()[: len(sample)],
            "pred_sev": p_rg,
        }
    ).head()
)